
<div style="text-align: center; background-color: lightblue; padding: 15px;">
    <p style="text-align: center;">بسم الله، والحمد لله، والصلاة والسلام على رسول الله وعلى آله وصحبه</p>
</div>

# THIS NOTEBOOKS AIMS TO EXPLORE THE LLMs  

In [5]:
import json

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

data = {
    "period": "Janvier 2026",
    "temperature": {
        "mean": 15.8,
        "anomaly": 1.7
    },
    "precipitation": {
        "accumulated": 42.5,
        "reference": 68.3,
        "anomaly_percent": -37.8
    },
    "extreme_temperature": {
        "maximum": 29.4,
        "minimum": 2.8
    }
}



print(f"Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)

model.eval()


json_data = json.dumps(
data,
ensure_ascii=False,
indent=2
        )

prompt = f"""
You are a professional meteorologist working for a meteorological service.

Your task is to transform verified meteorological statistics into a
scientifically accurate monthly meteorological report.

IMPORTANT RULES:

1. Do not invent numerical values.
2. Use only the information provided in the JSON.
3. Do not perform calculations unless absolutely necessary.
4. Clearly distinguish observations from interpretations.
5. Use professional meteorological terminology.
6. Write the report in French.
7. Produce a clear and concise report.
8. If information is missing, do not invent it.

Meteorological data:

{json_data}

Write the meteorological report now.
"""

messages = [
            {
                "role": "system",
                "content": (
                    "You are an expert meteorologist and scientific "
                    "report writer."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ]

text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

inputs = tokenizer(
            text,
            return_tensors="pt"
        ).to(model.device)

with torch.no_grad():
    outputs =   model.generate(
                **inputs,
                max_new_tokens=1500,
                temperature=0.3,
                do_sample=True,
            )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

report = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        )






report

# report = generator.generate(data)

# print("\n")
# print("=" * 80)
# print("RAPPORT METEOROLOGIQUE")
# print("=" * 80)
# print(report)


Loading model: Qwen/Qwen3-4B-Instruct-2507


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 398/398 [00:00<00:00, 7271.41it/s]


'**Rapport météorologique mensuel – Janvier 2026**\n\n**Période :** Janvier 2026\n\n**Température :**  \nLa température moyenne mensuelle s’est élevée à 15,8\u202f°C, enregistrant un écart par rapport à la moyenne climatique de +1,7\u202f°C. Ce surplus thermique indique une période légèrement plus chaude que la norme.\n\n**Précipitations :**  \nL’accumulation mensuelle des précipitations s’est élevée à 42,5\u202fmm, contre une valeur de référence de 68,3\u202fmm. L’anomalie relative s’élève à -37,8\u202f%, ce qui signifie une déficience significative des précipitations par rapport à la moyenne historique. Cette situation suggère une sécheresse observée au cours du mois.\n\n**Extremes :**  \nLa température maximale enregistrée a atteint 29,4\u202f°C, tandis que la température minimale a chuté à 2,8\u202f°C. Ces valeurs extrêmes reflètent une variation diurne de 26,6\u202f°C, typique d’un climat à forte amplitude thermique.\n\n**Conclusion :**  \nJanvier 2026 a été marqué par une tempéra